# DTW Experiments — Algorithm Documentation

## Algorithm: Local DTW via Smith-Waterman (`local_dtw_alignment`)

### What it finds — formal definition

Given two multivariate time-series matrices $A \in \mathbb{R}^{n \times d}$ and $B \in \mathbb{R}^{m \times d}$, and a scalar `step_threshold` $\tau \in \mathbb{R}$, the algorithm finds:

> **The pair of contiguous sub-sequences (infixes) $A[i_0:i_1]$ and $B[j_0:j_1]$ whose DTW alignment maximises the total accumulated reward $S^*$.**

The reward at each aligned pair $(i, j)$ is:
$$s(i,j) = \tau - \|A_i - B_j\|_2$$

A pair earns a **positive reward** when the Euclidean distance between the two feature vectors is smaller than $\tau$, and a **penalty** when it is larger. The total score is the sum of per-step rewards along the alignment path.

Formally, $S^* = \max_{i,j} \, \text{DP}[i][j]$, where the DP table is defined by:

$$\text{DP}[i][j] = \max\!\Big(0,\; \text{DP}[i-1][j-1] + s(i,j),\; \text{DP}[i-1][j] - g + s(i,j),\; \text{DP}[i][j-1] - g + s(i,j)\Big)$$

with boundary condition $\text{DP}[0,\cdot] = \text{DP}[\cdot,0] = 0$ and gap penalty $g \ge 0$.

The result is the triple $(S^*, i_0:i_1, j_0:j_1)$ recovered by traceback from the cell that achieved $S^*$.

---

### How it works — step by step

1. **Similarity matrix** — compute $\text{sim}[i][j] = \tau - \|A_i - B_j\|_2$ for all $(i,j)$. This is a signed reward: positive iff the two frames are within distance $\tau$.

2. **Smith-Waterman DP** — fill the $(n+1)\times(m+1)$ score table using the recurrence above. Three moves are allowed, corresponding to standard DTW warping operations:
   - *Diagonal* $(i-1,j-1)$: advance both sequences simultaneously — no gap.
   - *Vertical* $(i-1,j)$: skip one step in $A$ while staying in $B$ — gap in $B$.
   - *Horizontal* $(i,j-1)$: skip one step in $B$ while staying in $A$ — gap in $A$.
   
   The $\max(\cdot, 0)$ floor resets the score to zero whenever the accumulated match becomes negative, which allows the alignment to **start fresh** at any position — this is what makes it *local* (infix) rather than global.

3. **Best cell** — the cell $(i^*, j^*)$ with the maximum score $S^*$ marks the **end** of the best-matching infix pair.

4. **Traceback** — follow the stored predecessor pointers backwards from $(i^*, j^*)$ until a zero cell is reached. The visited cells trace out the DTW warping path; the indices give the infix bounds $[i_0, i^*]$ in $A$ and $[j_0, j^*]$ in $B$.

---

### Role of `step_threshold` $\tau$

$\tau$ is the **distance budget per alignment step**.  
- If $\tau$ is too small (below the typical pairwise EEG distance), nearly every step yields a negative reward and the algorithm can never accumulate a positive score → $S^* \approx 0$ for all pairs, making discrimination impossible.  
- If $\tau$ is too large, every pair earns a positive reward regardless of actual similarity → scores saturate and all trials look equally similar.  
- The calibration cell (cell 11) measures empirical pairwise EEG distances to inform a good $\tau$ range, and the sweep cells (cells 12, 14) search for the best $\tau$ on a validation set.

---

### Usage in the experiments

`dtw_match_accuracy(query_ds, ref_ds, step_threshold)` iterates over every element of `query_ds`, computes $S^*$ against every element of `ref_ds`, and **predicts the `trial_id` of the reference with the highest score**. Accuracy is the fraction of queries where the predicted `trial_id` matches the true one.


In [1]:
import numpy as np
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw

# Define two matrices: shape = (Sequence_Length, Number_of_Features)
# We are warping along the Sequence_Length dimension (axis 0).
# The matrices can have different sequence lengths, but MUST have the same number of features.
matrix_A = np.array([
    [1.0, 2.0],
    [1.5, 2.5],
    [2.0, 3.0],
    [5.0, 5.0]
]) # Shape: (4, 2)

matrix_B = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [2.5, 3.5],
    [5.0, 5.0],
    [5.1, 5.1]
]) # Shape: (5, 2)

# Calculate the approximate DTW distance
# - dist=euclidean: tells the algorithm how to compare the 1D feature arrays at each step.
# - radius: controls the approximation. A larger radius increases exactness but takes more computation time.
distance, warp_path = fastdtw(matrix_A, matrix_B, dist=euclidean, radius=1)

print(f"Approximate DTW Distance: {distance}")
print(f"Warping Path (Index mapping between A and B): \n{warp_path}")

Approximate DTW Distance: 1.5556349186104041
Warping Path (Index mapping between A and B): 
[(0, 0), (1, 1), (2, 1), (2, 2), (3, 3), (3, 4)]


In [2]:
import numpy as np
from tslearn.metrics import dtw_subsequence_path

# Matrix A: The shorter "Query" sequence
# Shape: (Sequence_Length=3, Features=2)
matrix_A = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 4.0]
]) 

# Matrix B: The longer "Reference" sequence
# Notice that an approximate version of matrix_A is hidden in the middle.
# Shape: (Sequence_Length=7, Features=2)
matrix_B = np.array([
    [9.0, 9.0], # Unmatched prefix
    [8.0, 8.0], # Unmatched prefix
    [1.1, 2.1], # <--- Match starts roughly here
    [2.0, 2.9], # <--- Match continues
    [3.1, 4.2], # <--- Match ends here
    [9.0, 9.0], # Unmatched suffix
    [8.0, 8.0]  # Unmatched suffix
])

# dtw_subsequence_path aligns the entirety of matrix_A to a sub-segment of matrix_B
# It returns the optimal path and the exact distance for that specific segment.
path, distance = dtw_subsequence_path(matrix_A, matrix_B)

print(f"Subsequence DTW Distance: {distance:.4f}")
print("Warping Path (Index of A -> Index of B):")
for step in path:
    print(f"A[{step[0]}] aligns with B[{step[1]}]")
    

Subsequence DTW Distance: 0.2828
Warping Path (Index of A -> Index of B):
A[0] aligns with B[2]
A[1] aligns with B[3]
A[2] aligns with B[4]


In [3]:
import numpy as np
from scipy.spatial.distance import cdist

def local_dtw_alignment(matrix_A, matrix_B, step_threshold, gap_penalty=0.0):
    """
    Finds the longest infix of A that matches an infix of B, based on a cost threshold.
    """
    # 1. Calculate all pairwise Euclidean distances between vectors in A and B
    dist_matrix = cdist(matrix_A, matrix_B, metric='euclidean')
    
    # 2. Convert Distances to Similarity Scores
    # Distances strictly smaller than the threshold become positive rewards.
    # Distances larger than the threshold become negative penalties.
    sim_matrix = step_threshold - dist_matrix
    
    n, m = len(matrix_A), len(matrix_B)
    
    # DP matrix to store scores, plus an extra row/col of zeros for boundaries
    score_matrix = np.zeros((n + 1, m + 1))
    
    # To remember where we came from to reconstruct the path
    traceback = np.zeros((n + 1, m + 1, 2), dtype=int)
    
    max_score = 0
    max_pos = (0, 0)
    
    # 3. Dynamic Programming (Smith-Waterman style)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Calculate scores for moving diagonally (match), down, or right (gaps/warps)
            match = score_matrix[i-1, j-1] + sim_matrix[i-1, j-1]
            delete = score_matrix[i-1, j] - gap_penalty + sim_matrix[i-1, j-1]
            insert = score_matrix[i, j-1] - gap_penalty + sim_matrix[i-1, j-1]
            
            # The core of Local Alignment: If the score drops below 0, reset it.
            # This allows the algorithm to start a brand new infix match anywhere.
            best_score = max(0, match, delete, insert)
            score_matrix[i, j] = best_score
            
            # Record the move for our traceback
            if best_score == 0:
                traceback[i, j] = (0, 0) # Path broken/reset
            elif best_score == match:
                traceback[i, j] = (i-1, j-1)
            elif best_score == delete:
                traceback[i, j] = (i-1, j)
            else:
                traceback[i, j] = (i, j-1)
                
            # Keep track of the highest score found anywhere in the matrix
            if best_score > max_score:
                max_score = best_score
                max_pos = (i, j)
                
    # 4. Traceback to extract the actual path indices
    path = []
    curr_i, curr_j = max_pos
    
    # Work backwards from the max score until we hit a 0 (the start of the local match)
    while score_matrix[curr_i, curr_j] > 0:
        path.append((curr_i - 1, curr_j - 1)) # Adjust for the +1 offset in DP matrix
        curr_i, curr_j = traceback[curr_i, curr_j]
        
    path.reverse() # Reverse to get the path from start to end
    return path, max_score

# --- Example Usage ---
# Let's say we have two noisy sequences, but they share a similar segment in the middle.
matrix_A = np.array([[10, 10], [1, 2], [2, 3], [3, 4], [20, 20]]) # Infix is indices 1 to 3
matrix_B = np.array([[99, 99], [88, 88], [1.1, 2.1], [2.2, 2.8], [3.1, 4.2], [77, 77]]) # Infix is indices 2 to 4

# step_threshold of 1.0 means any distance < 1.0 per step is a "good" match.
path, score = local_dtw_alignment(matrix_A, matrix_B, step_threshold=1.0, gap_penalty=0.1)

print("Warping Path (Index of A -> Index of B):")
for step in path:
    print(f"A[{step[0]}] -> B[{step[1]}]")

Warping Path (Index of A -> Index of B):
A[1] -> B[2]
A[2] -> B[3]
A[3] -> B[4]


In [4]:
from eeg_music.data import EEGMusicDataset
from pathlib import Path
# ds = EEGMusicDataset.load_ondisk(Path("./datasets/bcmi_preprocessed/bcmi_full_ica_40ch/"))
train_ds = EEGMusicDataset.load_ondisk(Path("./datasets/musing_preprocessed/train_onesubj_musing_ica_40ch/"))
test_ds = EEGMusicDataset.load_ondisk(Path("./datasets/musing_preprocessed/test_onesubj_musing_ica_40ch/"))
len(train_ds), len(test_ds)

/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


(12, 12)

In [5]:
from fractions import Fraction
from eeg_music.data import ArrayStratifiedSamplingDataset
from eeg_music.data import MappedDataset
from eeg_music.onset_conversion import trial_wavraw_to_noteonsets

def after_loaded(ds):
  # ds = MappedDataset(ds, trial_wavraw_to_noteonsets)
  return ArrayStratifiedSamplingDataset(ds, 10, trial_length_secs=Fraction(1, 1))

[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


In [6]:
train_ds_strat = after_loaded(train_ds)
test_ds_strat = after_loaded(test_ds)

In [7]:
train_ds[3].eeg_data.get_array().data.shape
train_ds[4].trial_id

'song_05'

In [8]:
train_ds_strat[3].trial_id

'song_01'

In [9]:
import numpy as np
from scipy.spatial.distance import cdist

def local_dtw_alignment_v2(matrix_A, matrix_B, step_threshold, gap_penalty=0.0):
    """Local DTW via Smith-Waterman: finds best-matching infixes of A and B."""
    sim_matrix = step_threshold - cdist(matrix_A, matrix_B, metric='euclidean')
    n, m = len(matrix_A), len(matrix_B)
    score_matrix = np.zeros((n + 1, m + 1))
    traceback = np.zeros((n + 1, m + 1, 2), dtype=int)
    max_score, max_pos = 0, (0, 0)

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = sim_matrix[i-1, j-1]
            match = score_matrix[i-1, j-1] + s
            delete = score_matrix[i-1, j] - gap_penalty + s
            insert = score_matrix[i, j-1] - gap_penalty + s
            best = max(0, match, delete, insert)
            score_matrix[i, j] = best
            if best == 0:
                traceback[i, j] = (0, 0)
            elif best == match:
                traceback[i, j] = (i-1, j-1)
            elif best == delete:
                traceback[i, j] = (i-1, j)
            else:
                traceback[i, j] = (i, j-1)
            if best > max_score:
                max_score, max_pos = best, (i, j)

    path = []
    ci, cj = max_pos
    while score_matrix[ci, cj] > 0:
        path.append((ci - 1, cj - 1))
        ci, cj = traceback[ci, cj]
    path.reverse()
    return path, max_score


def local_dtw_checked(matrix_A, matrix_B, step_threshold, gap_penalty=0.0):
    """Run both implementations and assert they agree."""
    path1, score1 = local_dtw_alignment(matrix_A, matrix_B, step_threshold, gap_penalty)
    path2, score2 = local_dtw_alignment_v2(matrix_A, matrix_B, step_threshold, gap_penalty)
    if score1 != score2 or path1 != path2:
        raise AssertionError(
            f"Implementations disagree!\n"
            f"  v1: score={score1}, path_len={len(path1)}\n"
            f"  v2: score={score2}, path_len={len(path2)}"
        )
    return path1, score1


def get_eeg_matrix(trial):
    """Extract EEG as (samples, channels) matrix."""
    return trial.eeg_data.get_array().data.T


def dtw_match_accuracy(query_ds, ref_ds, step_threshold, gap_penalty=0.0):
    """
    For each element in query_ds, find the ref_ds element with highest
    local_dtw_alignment score, predict its trial_id, and compute accuracy.

    Works for both full-trial datasets and stratified snippet datasets.
    """
    ref_matrices = [(get_eeg_matrix(ref_ds[j]), ref_ds[j].trial_id) for j in range(len(ref_ds))]
    correct, total = 0, len(query_ds)
    predictions = []

    for i in range(total):
        query_trial = query_ds[i]
        q_mat = get_eeg_matrix(query_trial)
        true_id = query_trial.trial_id

        best_score, best_id = -1, None
        for r_mat, r_id in ref_matrices:
            _, score = local_dtw_checked(q_mat, r_mat, step_threshold, gap_penalty)
            if score > best_score:
                best_score, best_id = score, r_id

        hit = best_id == true_id
        correct += hit
        predictions.append((true_id, best_id, best_score, hit))
        print(f"[{i+1}/{total}] true={true_id}  pred={best_id}  score={best_score:.2f}  {'✓' if hit else '✗'}")

    accuracy = correct / total
    print(f"\nAccuracy: {correct}/{total} = {accuracy:.2%}")
    return accuracy, predictions

In [10]:
# Experiment 1: Full trial matching (train_ds -> test_ds)
step_threshold = 1e-4
print("=== Experiment 1: Full trial matching ===")
acc1, preds1 = dtw_match_accuracy(train_ds, test_ds, step_threshold)

=== Experiment 1: Full trial matching ===


[1/12] true=song_01  pred=song_01  score=0.00  ✓
[2/12] true=song_02  pred=song_01  score=0.00  ✗
[2/12] true=song_02  pred=song_01  score=0.00  ✗
[3/12] true=song_03  pred=song_01  score=0.00  ✗
[3/12] true=song_03  pred=song_01  score=0.00  ✗
[4/12] true=song_04  pred=song_01  score=0.00  ✗
[4/12] true=song_04  pred=song_01  score=0.00  ✗
[5/12] true=song_05  pred=song_01  score=0.00  ✗
[5/12] true=song_05  pred=song_01  score=0.00  ✗
[6/12] true=song_06  pred=song_01  score=0.00  ✗
[6/12] true=song_06  pred=song_01  score=0.00  ✗
[7/12] true=song_07  pred=song_01  score=0.00  ✗
[7/12] true=song_07  pred=song_01  score=0.00  ✗
[8/12] true=song_08  pred=song_01  score=0.00  ✗
[8/12] true=song_08  pred=song_01  score=0.00  ✗
[9/12] true=song_09  pred=song_01  score=0.00  ✗
[9/12] true=song_09  pred=song_01  score=0.00  ✗
[10/12] true=song_10  pred=song_01  score=0.00  ✗
[10/12] true=song_10  pred=song_01  score=0.00  ✗
[11/12] true=song_11  pred=song_01  score=0.00  ✗
[11/12] true=song

In [11]:
# Check typical pairwise distances to calibrate threshold range
from scipy.spatial.distance import cdist

a = get_eeg_matrix(train_ds[0])
b = get_eeg_matrix(test_ds[0])
dists = cdist(a[:200], b[:200], metric='euclidean')
print(f"EEG pairwise distance stats (first 200 samples of trial 0):")
print(f"  min={dists.min():.6f}  median={np.median(dists):.6f}  mean={dists.mean():.6f}  max={dists.max():.6f}")
print(f"  percentiles: 10%={np.percentile(dists,10):.6f}  25%={np.percentile(dists,25):.6f}  75%={np.percentile(dists,75):.6f}  90%={np.percentile(dists,90):.6f}")

EEG pairwise distance stats (first 200 samples of trial 0):
  min=0.086156  median=0.436747  mean=0.481859  max=1.835660
  percentiles: 10%=0.284595  25%=0.348835  75%=0.570799  90%=0.709777


In [12]:
# Sweep step_threshold on train_ds_strat vs test_ds
thresholds = [0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.8, 1.0]
results = {}

for th in thresholds:
    print(f"\n{'='*50}")
    print(f"step_threshold = {th}")
    print(f"{'='*50}")
    acc, preds = dtw_match_accuracy(train_ds_strat, test_ds, step_threshold=th)
    results[th] = acc

print(f"\n\n{'='*50}")
print("Summary:")
print(f"{'='*50}")
for th, acc in sorted(results.items()):
    bar = '█' * int(acc * 40)
    print(f"  threshold={th:<5}  accuracy={acc:.2%}  {bar}")
best_th = max(results, key=results.get)
print(f"\nBest threshold: {best_th}  ({results[best_th]:.2%})")


step_threshold = 0.1


[1/120] true=song_01  pred=song_01  score=0.00  ✓
[2/120] true=song_01  pred=song_01  score=0.00  ✓
[2/120] true=song_01  pred=song_01  score=0.00  ✓
[3/120] true=song_01  pred=song_01  score=0.00  ✓
[3/120] true=song_01  pred=song_01  score=0.00  ✓
[4/120] true=song_01  pred=song_01  score=0.00  ✓
[4/120] true=song_01  pred=song_01  score=0.00  ✓
[5/120] true=song_01  pred=song_01  score=0.00  ✓
[5/120] true=song_01  pred=song_01  score=0.00  ✓
[6/120] true=song_01  pred=song_01  score=0.00  ✓
[6/120] true=song_01  pred=song_01  score=0.00  ✓
[7/120] true=song_01  pred=song_01  score=0.00  ✓
[7/120] true=song_01  pred=song_01  score=0.00  ✓
[8/120] true=song_01  pred=song_01  score=0.00  ✓
[8/120] true=song_01  pred=song_01  score=0.00  ✓
[9/120] true=song_01  pred=song_01  score=0.01  ✓
[9/120] true=song_01  pred=song_01  score=0.01  ✓
[10/120] true=song_01  pred=song_01  score=0.00  ✓
[10/120] true=song_01  pred=song_01  score=0.00  ✓
[11/120] true=song_02  pred=song_02  score=8.34 

In [13]:
val_1_onesubj_ds = EEGMusicDataset.load_ondisk(Path("./datasets/musing_preprocessed/val_1_onesubj_musing_ica_40ch"))
test_1_onesubj_ds = EEGMusicDataset.load_ondisk(Path("./datasets/musing_preprocessed/test_1_onesubj_musing_ica_40ch"))

In [19]:
# Stratified versions of val and test
val_1_strat = after_loaded(val_1_onesubj_ds)
test_1_strat = after_loaded(test_1_onesubj_ds)

# Sweep threshold on val set (train_ds_strat queries vs val references)
thresholds_v2 = [0.1, 0.15, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.8, 1.0]
val_results = {}

for th in thresholds_v2:
    print(f"\n{'='*50}")
    print(f"[VAL] step_threshold = {th}")
    print(f"{'='*50}")
    acc, _ = dtw_match_accuracy(train_ds_strat, val_1_onesubj_ds, step_threshold=th)
    val_results[th] = acc

print(f"\n\n{'='*50}")
print("Val Summary:")
print(f"{'='*50}")
for th, acc in sorted(val_results.items()):
    bar = '█' * int(acc * 40)
    print(f"  threshold={th:<5}  accuracy={acc:.2%}  {bar}")
best_th_val = max(val_results, key=val_results.get)
print(f"\nBest val threshold: {best_th_val}  ({val_results[best_th_val]:.2%})")


[VAL] step_threshold = 0.1
[1/120] true=song_01  pred=song_01  score=1.00  ✓
[1/120] true=song_01  pred=song_01  score=1.00  ✓
[2/120] true=song_01  pred=song_01  score=1.01  ✓
[2/120] true=song_01  pred=song_01  score=1.01  ✓
[3/120] true=song_01  pred=song_01  score=1.03  ✓
[3/120] true=song_01  pred=song_01  score=1.03  ✓
[4/120] true=song_01  pred=song_01  score=1.01  ✓
[4/120] true=song_01  pred=song_01  score=1.01  ✓
[5/120] true=song_01  pred=song_01  score=1.01  ✓
[5/120] true=song_01  pred=song_01  score=1.01  ✓
[6/120] true=song_01  pred=song_01  score=1.12  ✓
[6/120] true=song_01  pred=song_01  score=1.12  ✓
[7/120] true=song_01  pred=song_01  score=1.00  ✓
[7/120] true=song_01  pred=song_01  score=1.00  ✓
[8/120] true=song_01  pred=song_01  score=1.01  ✓
[8/120] true=song_01  pred=song_01  score=1.01  ✓
[9/120] true=song_01  pred=song_01  score=1.31  ✓
[9/120] true=song_01  pred=song_01  score=1.31  ✓
[10/120] true=song_01  pred=song_01  score=1.06  ✓
[10/120] true=song_01

In [20]:
# Evaluate best val threshold on test set
print(f"Best threshold from val: {best_th_val}")
print(f"\n{'='*50}")
print(f"[TEST] step_threshold = {best_th_val}")
print(f"{'='*50}")
test_acc, test_preds = dtw_match_accuracy(train_ds_strat, test_1_onesubj_ds, step_threshold=best_th_val)
print(f"\nFinal test accuracy with threshold={best_th_val}: {test_acc:.2%}")

Best threshold from val: 0.1

[TEST] step_threshold = 0.1
[1/120] true=song_01  pred=song_01  score=0.00  ✓
[1/120] true=song_01  pred=song_01  score=0.00  ✓
[2/120] true=song_01  pred=song_01  score=0.00  ✓
[2/120] true=song_01  pred=song_01  score=0.00  ✓
[3/120] true=song_01  pred=song_01  score=0.00  ✓
[3/120] true=song_01  pred=song_01  score=0.00  ✓
[4/120] true=song_01  pred=song_01  score=0.00  ✓
[4/120] true=song_01  pred=song_01  score=0.00  ✓
[5/120] true=song_01  pred=song_01  score=0.00  ✓
[5/120] true=song_01  pred=song_01  score=0.00  ✓
[6/120] true=song_01  pred=song_01  score=0.00  ✓
[6/120] true=song_01  pred=song_01  score=0.00  ✓
[7/120] true=song_01  pred=song_01  score=0.00  ✓
[7/120] true=song_01  pred=song_01  score=0.00  ✓
[8/120] true=song_01  pred=song_01  score=0.00  ✓
[8/120] true=song_01  pred=song_01  score=0.00  ✓
[9/120] true=song_01  pred=song_01  score=0.00  ✓
[9/120] true=song_01  pred=song_01  score=0.00  ✓
[10/120] true=song_01  pred=song_01  score